# Learning Polars with `ticker_tape` data
**Primary objective:** learn **Polars (>=1.31.0)** by exploring your `ticker_tape.py` output dataset.

This notebook assumes you have already generated:

- `data_drops/universe_common.parquet`

If your file is elsewhere, update the path in the setup cell.

---

### What you'll learn (beginner → advanced)
- Reading Parquet + schema inspection
- Selecting, filtering, sorting, casting
- Expressions (`pl.col`, `when/then/otherwise`, string ops)
- Aggregations + `group_by`
- Joins + deduping strategies
- LazyFrames (`scan_parquet`) + query optimization
- Window functions and ranking
- Pivoting, sampling, and exporting
- Practical “data validation” checks you’ll reuse in modeling pipelines

At the end there’s a short **quiz** (with optional solutions).


In [1]:
# Setup (Polars >= 1.31.0)

import polars as pl
from pathlib import Path

DATA_PATH = Path("data_drops/universe_common.parquet")  # <-- change if needed

print("Polars:", pl.__version__)
assert tuple(map(int, pl.__version__.split(".")[:2])) >= (1, 31), "Please upgrade to polars>=1.31.0"

# Optional: nicer dataframe display in notebooks
pl.Config.set_tbl_rows(12)
pl.Config.set_tbl_cols(-1)
pl.Config.set_tbl_width_chars(140)
pl.Config.set_fmt_str_lengths(120)

DATA_PATH.exists(), str(DATA_PATH)


Polars: 1.31.0


(True, 'data_drops/universe_common.parquet')

## 1) Load the dataset (DataFrame vs LazyFrame)

Polars has two main modes:

- **DataFrame**: eager (executes immediately)
- **LazyFrame**: lazy (builds a query plan; executes when you `.collect()`)

Start eager so you can “touch the data,” then later we’ll switch to lazy for scalable workflows.


In [6]:
!ls /workspaces/smart_dev/projects/Notebooks/Stocks/flint/data

AAL.parquet	  GC_F.parquet			  NIO.parquet	 _TYX.parquet
AAPL.parquet	  HG_F.parquet			  ONCY.parquet	 _VIX9D.parquet
AMC.parquet	  HUYA.parquet			  processed	 _VIX.parquet
AMZN.parquet	  _IRX.parquet			  QS.parquet	 _VVIX.parquet
BTC_USD.parquet   IVR.parquet			  RUN.parquet	 XLC.parquet
CL_F.parquet	  KO.parquet			  RYCEY.parquet  XLI.parquet
DOGE_USD.parquet  LPTX.parquet			  SI_F.parquet	 XLK.parquet
DPST.parquet	  macro_indicators_daily.parquet  SPY.parquet	 XLRE.parquet
DX_Y_NYB.parquet  MO.parquet			  _TNX.parquet	 XLV.parquet
ETH_USD.parquet   MSFT.parquet			  TSLA.parquet	 XLY.parquet
FNGU.parquet	  NG_F.parquet			  TSLZ.parquet


In [ ]:
df = pl.read_parquet('~/projects/Notebooks/Stocks/flint')
df.head()

ticker,name,ticker_type,security_type,exchange,exchange_name,source
str,str,str,str,str,str,str
"""A""","""Agilent Technologies, Inc. Common Stock""","""listed""","""common""","""N""","""NYSE""","""nasdaqtrader"""
"""AA""","""Alcoa Corporation Common Stock ""","""listed""","""common""","""N""","""NYSE""","""nasdaqtrader"""
"""AABB""","""Asia Broadband Inc Common Stock""","""otc""","""common""","""U""","""OTC (CAT)""","""finra_cat"""
"""AACB""","""Artius II Acquisition Inc. - Class A Ordinary Shares""","""listed""","""common""","""Q""","""Nasdaq""","""nasdaqtrader"""
"""AACG""","""ATA Creativity Global - American Depositary Shares, each representing two common shares""","""listed""","""common""","""Q""","""Nasdaq""","""nasdaqtrader"""


In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv

# Try a few likely locations (choose what you want)
candidates = [
    Path.cwd() / ".env",                      # repo root if running from there
    Path("/workspaces") / ".env",             # less common
    Path.home() / ".config/myapp/.env",       # private location
]

for p in candidates:
    if p.exists():
        load_dotenv(dotenv_path=p, override=False)
        break

api_key = os.getenv("ALPHAVANTAGE_API_KEY", "")
if not api_key:
    raise RuntimeError("Missing ALPHAVANTAGE_API_KEY (env var or .env file).")


ADCL6PCWHWVMDP7S


In [3]:
df.schema

Schema([('ticker', String),
        ('name', String),
        ('ticker_type', String),
        ('security_type', String),
        ('exchange', String),
        ('exchange_name', String),
        ('source', String)])

### Quick expectations (sanity)
Your `ticker_tape` common universe typically includes columns like:

- `ticker` (string)
- `name` (string)
- `ticker_type` (`listed` or `otc`)
- `security_type` (`common`)
- `exchange` (letter code, including `U` for OTC)
- `exchange_name` (human label)
- `source` (`nasdaqtrader` or `finra_cat`)

Let's confirm uniques and row counts.


In [5]:
df.select(
    pl.len().alias("rows"),
    pl.n_unique("ticker").alias("unique_tickers"),
    pl.n_unique("ticker_type").alias("ticker_types"),
    pl.n_unique("exchange").alias("exchanges"),
    pl.n_unique("exchange_name").alias("exchange_names"),
    pl.n_unique("source").alias("sources"),
)

rows,unique_tickers,ticker_types,exchanges,exchange_names,sources
u32,u32,u32,u32,u32,u32
10021,10021,2,5,5,2


## 2) Basic exploration

### Selecting columns
Polars uses `select()` with expressions. `pl.col("name")` is the basic building block.


In [17]:
import pandas as pd
df_pd = df.to_pandas()
df_pd[["ticker","ticker_type","exchange","exchange_name"]].head(10)

,ticker,ticker_type,exchange,exchange_name
0,A,listed,N,NYSE
1,AA,listed,N,NYSE
2,AABB,otc,U,OTC (CAT)
3,AACB,listed,Q,Nasdaq
4,AACG,listed,Q,Nasdaq
5,AACS,otc,U,OTC (CAT)
6,AAGC,otc,U,OTC (CAT)
7,AAGH,otc,U,OTC (CAT)
8,AAGR,otc,U,OTC (CAT)
9,AAL,listed,Q,Nasdaq


In [ ]:
df.select(
    "ticker",
    "ticker_type",
    "exchange",
    "exchange_name"
).head(10)


ticker,ticker_type,exchange,exchange_name
str,str,str,str
"""A""","""listed""","""N""","""NYSE"""
"""AA""","""listed""","""N""","""NYSE"""
"""AABB""","""otc""","""U""","""OTC (CAT)"""
"""AACB""","""listed""","""Q""","""Nasdaq"""
"""AACG""","""listed""","""Q""","""Nasdaq"""
"""AACS""","""otc""","""U""","""OTC (CAT)"""
"""AAGC""","""otc""","""U""","""OTC (CAT)"""
"""AAGH""","""otc""","""U""","""OTC (CAT)"""
"""AAGR""","""otc""","""U""","""OTC (CAT)"""


### Sorting


In [18]:
df.select("ticker", "exchange", "ticker_type").sort("ticker").head(10)

ticker,exchange,ticker_type
str,str,str
"""A""","""N""","""listed"""
"""AA""","""N""","""listed"""
"""AABB""","""U""","""otc"""
"""AACB""","""Q""","""listed"""
"""AACG""","""Q""","""listed"""
"""AACS""","""U""","""otc"""
"""AAGC""","""U""","""otc"""
"""AAGH""","""U""","""otc"""
"""AAGR""","""U""","""otc"""


### Filtering (Boolean expressions)
Filters are expression-based. Combine with `&` and `|`.


In [19]:
df.filter(pl.col("ticker_type") == "listed").head(10)

ticker,name,ticker_type,security_type,exchange,exchange_name,source
str,str,str,str,str,str,str
"""A""","""Agilent Technologies, Inc. Common Stock""","""listed""","""common""","""N""","""NYSE""","""nasdaqtrader"""
"""AA""","""Alcoa Corporation Common Stock ""","""listed""","""common""","""N""","""NYSE""","""nasdaqtrader"""
"""AACB""","""Artius II Acquisition Inc. - Class A Ordinary Shares""","""listed""","""common""","""Q""","""Nasdaq""","""nasdaqtrader"""
"""AACG""","""ATA Creativity Global - American Depositary Shares, each representing two common shares""","""listed""","""common""","""Q""","""Nasdaq""","""nasdaqtrader"""
"""AAL""","""American Airlines Group, Inc. - Common Stock""","""listed""","""common""","""Q""","""Nasdaq""","""nasdaqtrader"""
"""AAM""","""AA Mission Acquisition Corp. Class A Ordinary Shares""","""listed""","""common""","""N""","""NYSE""","""nasdaqtrader"""
"""AAME""","""Atlantic American Corporation - Common Stock""","""listed""","""common""","""Q""","""Nasdaq""","""nasdaqtrader"""
"""AAMI""","""Acadian Asset Management Inc. Common Stock""","""listed""","""common""","""N""","""NYSE""","""nasdaqtrader"""
"""AAOI""","""Applied Optoelectronics, Inc. - Common Stock""","""listed""","""common""","""Q""","""Nasdaq""","""nasdaqtrader"""


In [20]:
# A quick look at OTC symbols
df.filter(pl.col("ticker_type") == "otc").select("ticker", "name", "exchange").head(10)

ticker,name,exchange
str,str,str
"""AABB""","""Asia Broadband Inc Common Stock""","""U"""
"""AACS""","""American Commerce Solutions, Inc. Common Stock""","""U"""
"""AAGC""","""All American Gold Corp. Common Stock""","""U"""
"""AAGH""","""America Great Health Common Stock""","""U"""
"""AAGR""","""AFRICAN AGRICULTURE HLDGS INC Common Stock""","""U"""
"""AAPI""","""Apple iSports Group, Inc. Common Stock""","""U"""
"""AAPJ""","""AAP, Inc. Common Stock""","""U"""
"""AAPT""","""All American Pet Company, Inc. Common Stock""","""U"""
"""AAQL""","""Antiaging Quantum Living Inc. Common Stock""","""U"""


## 3) Useful string operations

Polars has a strong string expression API under `.str`.
Let's do a few “data cleaning” style operations.


In [21]:
df.select(
    pl.col("ticker"),
    pl.col("name").str.len_chars().alias("name_len"),
    pl.col("name").str.contains("ETF", literal=True).alias("contains_ETF_literal"),
).head(10)


ticker,name_len,contains_ETF_literal
str,u32,bool
"""A""",39,false
"""AA""",31,false
"""AABB""",31,false
"""AACB""",52,false
"""AACG""",87,false
"""AACS""",46,false
"""AAGC""",36,false
"""AAGH""",33,false
"""AAGR""",42,false


### Regex filtering: likely foreign ADR-ish patterns
This is purely exploratory—your universe includes OTC too, so you might see ADR-like tickers.


In [25]:
# Example: tickers ending with 'Y' are often ADR tickers on OTC (not always).
df.filter(
    pl.col("ticker_type") == "otc").filter(
        pl.col("ticker").str.ends_with("Y")).select("ticker", "name", "ticker_type").head(15)

ticker,name,ticker_type
str,str,str
"""ABBY""","""Abby, Inc Common Stock""","""otc"""
"""AEGY""","""Alternative Energy Partners Inc. Common Stock""","""otc"""
"""AFFY""","""Affymax, Inc. Common Stock""","""otc"""
"""AGDY""","""Agri-Dynamics Inc Common Stock""","""otc"""
"""AGLY""","""Atlantis Glory Inc. Common Stock""","""otc"""
"""AGNY""","""Agavenny Corporation Common Stock""","""otc"""
…,…,…
"""ALRY""","""Allenergy Inc Common Stock""","""otc"""
"""AMGY""","""American Metal & Technology, Inc. Common Stock""","""otc"""


## 4) Grouping and aggregation

This is where Polars starts to feel *different* from pandas: all aggregation is expression-based.

We'll build a “counts dashboard” by exchange and ticker_type.


In [ ]:
counts = (
    df.group_by(["ticker_type", "exchange", "exchange_name", "source"])
      .agg(pl.len().alias("n"))
      .sort("n", descending=True)
)

counts.head(25)


In [ ]:
# Total counts by ticker_type
df.group_by("ticker_type").agg(pl.len().alias("n")).sort("n", descending=True)


### Pivot (wide table)
Pivot is useful for quick “matrix” views.


In [ ]:
wide = (
    df.group_by(["exchange", "exchange_name", "ticker_type"])
      .agg(pl.len().alias("n"))
      .pivot(values="n", index=["exchange", "exchange_name"], columns="ticker_type")
      .fill_null(0)
      .sort("exchange")
)
wide


## 5) Creating new columns

Use `with_columns()` to add derived fields.


In [ ]:
df2 = df.with_columns(
    pl.col("ticker").str.len_chars().alias("ticker_len"),
    pl.col("name").str.to_lowercase().alias("name_lower"),
    pl.when(pl.col("ticker_type") == "otc").then(pl.lit(True)).otherwise(pl.lit(False)).alias("is_otc"),
)

df2.select("ticker", "ticker_len", "is_otc", "name").head(10)


### Practical validation checks
These are the kinds of checks you'll want before scraping OHLCV.

- Are tickers unique?
- Any null tickers?
- Any unexpected exchange codes?


In [ ]:
validation = df.select(
    pl.len().alias("rows"),
    pl.col("ticker").is_null().sum().alias("null_ticker_count"),
    (pl.len() - pl.n_unique("ticker")).alias("duplicate_ticker_count"),
    pl.col("exchange").unique().sort().alias("exchange_codes_present"),
)
validation


## 6) Joins and deduping strategies

Even though your Parquet is already deduped, it's useful to understand joins because your next step (OHLCV ingestion) will almost certainly create additional tables you want to join back.

We'll create a tiny synthetic OHLCV “status” table and join it to your universe.


In [ ]:
# Fake example: pretend some tickers failed OHLCV retrieval
sample = df.select("ticker").sample(n=25, with_replacement=False, seed=7)
status = sample.with_columns(
    pl.when(pl.arange(0, pl.len()) % 5 == 0).then(pl.lit("fail")).otherwise(pl.lit("ok")).alias("ohlcv_status")
)

joined = df.join(status, on="ticker", how="left")
joined.select("ticker", "ticker_type", "exchange", "ohlcv_status").head(20)


Now: count failure rates by ticker_type. This is exactly the kind of diagnostic you’ll do in your pipeline.


In [ ]:
joined.group_by("ticker_type").agg(
    pl.len().alias("n_total"),
    (pl.col("ohlcv_status") == "fail").sum().alias("n_fail"),
).with_columns(
    (pl.col("n_fail") / pl.col("n_total")).alias("fail_rate")
).sort("fail_rate", descending=True)


## 7) LazyFrames (the scalable way)

When data gets big, prefer lazy:

- `scan_parquet()` builds a query plan
- `collect()` runs it
- `explain()` shows the plan

This matters a ton when you start creating multi-year OHLCV datasets.


In [ ]:
lf = pl.scan_parquet(DATA_PATH)
lf


In [ ]:
# Example lazy query: count by exchange and ticker_type
lazy_counts = (
    lf.group_by(["ticker_type", "exchange"])
      .agg(pl.len().alias("n"))
      .sort("n", descending=True)
)

# show the plan
print(lazy_counts.explain())

# execute
lazy_counts.collect()


### Lazy query optimizations you get “for free”
Polars can push down projections and predicates, meaning it only reads what it needs.

Try selecting only a couple columns and filtering: it's fast even on huge datasets.


In [ ]:
(
    pl.scan_parquet(DATA_PATH)
      .select(["ticker", "ticker_type", "exchange"])
      .filter(pl.col("ticker_type") == "listed")
      .group_by("exchange")
      .agg(pl.len().alias("n"))
      .sort("n", descending=True)
      .collect()
)


## 8) Window functions (advanced)

Window functions let you compute “group-aware” results without collapsing the table.

Example: rank tickers by name length **within each exchange**.


In [ ]:
ranked = (
    df.with_columns(
        pl.col("name").str.len_chars().alias("name_len"),
        pl.col("name").str.len_chars().rank(method="dense", descending=True).over("exchange").alias("name_len_rank_in_exchange"),
    )
    .select("ticker", "exchange", "ticker_type", "name_len", "name_len_rank_in_exchange", "name")
    .sort(["exchange", "name_len_rank_in_exchange"])
)

ranked.head(25)


## 9) Practical export patterns

You already export Parquet from `ticker_tape.py`. Here are common patterns you’ll use next:

- Export a filtered subset (e.g., listed-only)
- Export a “worklist” of tickers for scraping


In [ ]:
# Create a worklist of tickers (example: listed only)
worklist = (
    df.filter(pl.col("ticker_type") == "listed")
      .select("ticker")
      .unique()
      .sort("ticker")
)

worklist.head(10), worklist.height


In [ ]:
# Optional: write a smaller Parquet for scraping (or CSV if you want)
out_dir = Path("data_drops")
out_dir.mkdir(exist_ok=True)

worklist_path = out_dir / "worklist_listed_tickers.parquet"
worklist.write_parquet(worklist_path, compression="zstd", statistics=True)

worklist_path


## 10) Your next step: OHLCV scraping prep (minimal)

Before scraping, you usually want:
- a stable ticker list (the worklist)
- light stratification (listed vs OTC)
- maybe chunking

Here’s a simple chunking approach in Polars.


In [ ]:
# Chunk tickers into batches of 500 for OHLCV scraping
BATCH_SIZE = 500

work = df.select("ticker", "ticker_type", "exchange").sort("ticker")
work = work.with_row_index("row_idx")

chunks = (
    work.with_columns((pl.col("row_idx") // BATCH_SIZE).alias("batch_id"))
        .drop("row_idx")
)

chunks.group_by("batch_id").agg(pl.len().alias("n")).head(10)


# Quiz (beginner → advanced)

Try these without scrolling up too much. The point is to practice Polars expressions.

**Q1 (Beginner):** How many tickers are OTC vs listed? Return a 2-row table.

**Q2 (Beginner+):** List the top 15 exchanges by count of tickers.

**Q3 (Intermediate):** Create a DataFrame of tickers whose `name` contains “TRUST” (case-insensitive). Show 20 rows.

**Q4 (Intermediate):** Add a column `is_adr_like` where ticker ends with `Y`. Then compute the % of ADR-like tickers by `ticker_type`.

**Q5 (Advanced):** Using a LazyFrame, compute counts by (`ticker_type`, `exchange`) and show the query plan (`.explain()`).

---

Write your answers in the empty cells below.

(There are optional solutions after the blank cells.)


In [ ]:
# Q1
# your code here


In [ ]:
# Q2
# your code here


In [ ]:
# Q3
# your code here


In [ ]:
# Q4
# your code here


In [ ]:
# Q5
# your code here


## Optional solutions (peek only after you try)

If you want to actually “quiz yourself,” collapse this section in your notebook UI.


In [ ]:
# --- Solutions ---

# Q1
sol1 = df.group_by("ticker_type").agg(pl.len().alias("n")).sort("n", descending=True)
sol1


In [ ]:
# Q2
sol2 = df.group_by(["exchange", "exchange_name"]).agg(pl.len().alias("n")).sort("n", descending=True).head(15)
sol2


In [ ]:
# Q3
sol3 = df.filter(pl.col("name").str.to_lowercase().str.contains("trust", literal=True)).head(20)
sol3


In [ ]:
# Q4
tmp = df.with_columns(
    pl.col("ticker").str.ends_with("Y").alias("is_adr_like")
)
sol4 = (
    tmp.group_by("ticker_type")
       .agg(
           pl.len().alias("n"),
           pl.col("is_adr_like").sum().alias("n_adr_like"),
       )
       .with_columns((pl.col("n_adr_like") / pl.col("n")).alias("pct_adr_like"))
)
sol4


In [ ]:
# Q5
lf = pl.scan_parquet(DATA_PATH)
sol5 = (
    lf.group_by(["ticker_type", "exchange"])
      .agg(pl.len().alias("n"))
      .sort("n", descending=True)
)
print(sol5.explain())
sol5.collect()
